# Notebook 05 — Machine Learning on Market Data

**FIN 4600 · Lab 3 · Financial Data Analytics**

Duran, *Financial Services Technology* (3rd ed.), **Chapter 6 — Data Analytics**

---

Notebook 04 applied machine learning to a problem where the signal was real.
This notebook applies the *same machinery* to market prediction, where it
mostly is not — and that contrast is the lesson.

We do two things:

- **Task A: predict next month's direction.** This is what everyone tries
  first. We will do it carefully and watch it fail, and we will learn how to
  tell a failure from a success, which is harder than it sounds.
- **Task B: predict next month's volatility.** Same features, same models,
  different target — and here there is a real, if modest, edge. Understanding
  *why* the two differ is the point of the notebook.

> **Prerequisite:** run `03_features_and_targets.ipynb` first. It writes the
> file this notebook reads.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=None):
    """Return the repository root — the folder that contains data/sp500_prices.csv."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "sp500_prices.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "
        "and open the mtu4600-lab3-analytics folder itself, then re-run."
    )


REPO = find_repo_root()
DATA = REPO / "data"
plt.style.use(REPO / "fin4600.mplstyle")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)

RANDOM_STATE = 42

In [ ]:
features_path = DATA / "processed" / "monthly_features.csv"

if not features_path.exists():
    raise FileNotFoundError(
        "monthly_features.csv is missing.\n"
        "Run 03_features_and_targets.ipynb from top to bottom first — its last "
        "cell writes the file this notebook needs."
    )

data = pd.read_csv(features_path, parse_dates=["date"])
data["month"] = pd.PeriodIndex(data["month"], freq="M")
data = data.sort_values(["month", "ticker"]).reset_index(drop=True)

FEATURES = [
    "momentum_1m", "momentum_3m", "momentum_6m", "momentum_12m",
    "volatility_21d", "volatility_63d",
    "px_vs_200d_ma", "px_vs_52w_high", "volume_zscore",
    "vix_close", "vix_change_5d", "vix_vs_63d_avg",
]

print(f"{len(data):,} rows, {data['ticker'].nunique()} tickers, "
      f"{data['month'].nunique()} months")
print(f"{data['month'].min()} to {data['month'].max()}")
data.head(3)

## 1. Task A — predicting direction

Target: `next_month_up`, 1 if next month's return is positive.

The benchmark to beat is the **base rate**: always predicting "up". Anything
that does not beat it is worse than useless, because it costs money to run.

In [ ]:
base_rate = data["next_month_up"].mean()
print(f"Base rate — share of months that were up: {base_rate:.1%}")
print(f"So 'always predict up' scores {base_rate:.1%} accuracy.")

### Walk-forward validation

A single train/test split wastes data and gives one noisy number.
`TimeSeriesSplit` does better: it trains on an expanding window of history and
tests on the block that follows, repeatedly. Every test block is strictly in
the future relative to its training data.

This is the honest analogue of cross-validation for time series, and it is
what a research group would actually use.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

months = np.sort(data["month"].unique())
splitter = TimeSeriesSplit(n_splits=5)

print("Fold structure (by month, never overlapping, always forward):\n")
for fold, (train_index, test_index) in enumerate(splitter.split(months), start=1):
    train_months, test_months = months[train_index], months[test_index]
    print(f"  Fold {fold}:  train {train_months[0]} to {train_months[-1]}"
          f"  ({len(train_months):>2} months)"
          f"   ->  test {test_months[0]} to {test_months[-1]}"
          f"  ({len(test_months)} months)")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

classifiers = {
    "Always predict up": None,
    "Logistic regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

rows = []
predictions = []

for name, estimator in classifiers.items():
    for fold, (train_index, test_index) in enumerate(splitter.split(months), start=1):
        train = data[data["month"].isin(months[train_index])]
        test = data[data["month"].isin(months[test_index])]

        y_test = test["next_month_up"].to_numpy()

        if estimator is None:
            y_hat = np.ones_like(y_test)
            auc = np.nan
        else:
            pipeline = Pipeline(
                [("scale", StandardScaler()), ("model", estimator)]
            ).fit(train[FEATURES], train["next_month_up"])
            probability = pipeline.predict_proba(test[FEATURES])[:, 1]
            y_hat = (probability >= 0.5).astype(int)
            auc = roc_auc_score(y_test, probability)
            predictions.append(
                test[["month", "ticker", "next_month_return", "next_month_up"]]
                .assign(model=name, probability=probability, fold=fold)
            )

        rows.append(
            {
                "model": name,
                "fold": fold,
                "accuracy": accuracy_score(y_test, y_hat),
                "auc": auc,
                "share_predicted_up": y_hat.mean(),
            }
        )

fold_results = pd.DataFrame(rows)
summary = fold_results.groupby("model").agg(
    mean_accuracy=("accuracy", "mean"),
    sd_accuracy=("accuracy", "std"),
    mean_auc=("auc", "mean"),
    share_predicted_up=("share_predicted_up", "mean"),
)
summary.round(3)

Read that table slowly, because it is the whole point of the notebook.

- The models' mean accuracy is close to, and often below, "always predict up".
- AUC hovers near 0.5, which is the value of a coin flip.
- `share_predicted_up` shows what the models are actually doing: leaning
  heavily toward "up" — around two thirds of predictions — without leaning in
  a way that tracks the outcome. They have partly learned the base rate and
  have found nothing beyond it.

A model that has learned only the base rate can still post a respectable
accuracy score while containing no information at all. That is why AUC, which
is insensitive to the base rate, is the number to read first.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.5))
positions = np.arange(fold_results["fold"].nunique())
width = 0.26

for offset, (name, group) in zip([-width, 0, width], fold_results.groupby("model")):
    ax.bar(positions + offset, group.sort_values("fold")["accuracy"],
           width=width, label=name)

ax.axhline(base_rate, color="#52514e", linestyle="--", linewidth=1.2)
ax.annotate(f"overall base rate {base_rate:.0%}", xy=(-0.45, base_rate),
            xytext=(0, 5), textcoords="offset points", ha="left",
            fontsize=9, color="#52514e")
ax.set_xticks(positions, [f"Fold {f}" for f in sorted(fold_results['fold'].unique())])
ax.set_ylabel("Accuracy")
ax.set_title("Direction prediction: neither model reliably clears the base rate")
ax.set_ylim(0, 0.95)
ax.legend(loc="upper center", ncols=3, fontsize=9, bbox_to_anchor=(0.5, 1.0))
ax.grid(axis="x", visible=False)
plt.show()

### Is the difference statistically meaningful?

Suppose one model had come out two points above the base rate. Would that be
a discovery? Compute the standard error and see how wide the uncertainty is.

For a proportion, $\text{SE} = \sqrt{p(1-p)/n}$.

In [ ]:
n_test = int(len(data) / 6)  # roughly the size of one test block
p = base_rate
standard_error = np.sqrt(p * (1 - p) / n_test)

print(f"Test block size: about {n_test} observations")
print(f"Standard error of an accuracy estimate: {standard_error:.3f}")
print(f"A 95% interval around the base rate runs "
      f"{p - 1.96 * standard_error:.3f} to {p + 1.96 * standard_error:.3f}")
print()
print("Any accuracy inside that band is indistinguishable from guessing.")
print("Every number in the table above sits inside it.")

### Does it make money? The only test that matters

Accuracy is not the objective. Build the portfolio the model implies — each
month, hold an equal-weight basket of the names it scores highest — and
compare it with simply holding all 30.

In [ ]:
predicted = pd.concat(predictions, ignore_index=True)
forest_predictions = predicted[predicted["model"] == "Random forest"]

# Each month, go long the top third by predicted probability
def monthly_selection(group, fraction=1 / 3):
    k = max(1, int(len(group) * fraction))
    return group.nlargest(k, "probability")["next_month_return"].mean()


strategy = forest_predictions.groupby("month").apply(
    monthly_selection, include_groups=False
).rename("strategy")
benchmark = forest_predictions.groupby("month")["next_month_return"].mean().rename("equal_weight")

backtest = pd.concat([strategy, benchmark], axis=1).dropna()
backtest.index = backtest.index.to_timestamp()

cumulative = (1 + backtest).cumprod()

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.plot(cumulative.index, cumulative["equal_weight"], label="Hold all 30 equally",
        color="#2a78d6")
ax.plot(cumulative.index, cumulative["strategy"], label="Model's top third each month",
        color="#eb6834")
ax.axhline(1.0, color="#52514e", linewidth=0.8, linestyle="--")
ax.set_title("Out-of-sample backtest, before any trading costs")
ax.set_ylabel("Growth of $1")
ax.set_xlabel("")
ax.legend(loc="upper left")
plt.show()

annualised = (cumulative.iloc[-1] ** (12 / len(backtest)) - 1)
volatility = backtest.std() * np.sqrt(12)
print(f"{'':<28}{'Ann. return':>14}{'Ann. vol':>12}{'Return/vol':>12}")
for column in backtest.columns:
    print(f"{column:<28}{annualised[column]:>13.1%}{volatility[column]:>12.1%}"
          f"{annualised[column] / volatility[column]:>12.2f}")

print("\nAnd this ignores commissions, bid-ask spread, and market impact.")
print("A monthly-rebalanced strategy would pay all three, twelve times a year.")

### Why this is the expected result, not a bug

Prices already reflect what is publicly known. A pattern visible in five
years of daily closes, using features any undergraduate can compute, has been
arbitraged away by people with better data, faster systems, and more capital.

What firms that *do* extract signal from markets actually have:

- data nobody else has (order flow, satellite imagery, card transactions)
- a speed advantage measured in microseconds
- a structural role that earns a spread rather than a forecast
- a genuine risk premium they are paid to bear

None of those is a random forest on public closing prices.

**The professionally valuable skill here is recognising a null result and
saying so.** A student who reports "I tried and it does not work, here is the
evidence" has learned more than one who tortures the data until it confesses.

### Exercise 1

Rerun the fold loop with `min_samples_leaf=1` on the random forest — a model
free to memorise. Training accuracy will go up sharply. What happens to test
accuracy, and what is that phenomenon called?

In [ ]:
# YOUR CODE HERE

## 2. Task B — predicting volatility

Same features. Same validation. Different target: next month's realised
volatility.

The economic reason to expect a different answer: **direction is competed
away, but risk is persistent.** If a stock has been volatile for a month, it
is likely to be volatile next month. That is not an arbitrage opportunity —
you cannot buy last month's volatility — so nobody trades it away.

In [ ]:
volatility_data = data.dropna(subset=["next_month_volatility"]).copy()
print(f"{len(volatility_data):,} rows with a volatility target")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(volatility_data["volatility_21d"], volatility_data["next_month_volatility"],
           s=12, alpha=0.3, color="#2a78d6", edgecolor="none")
limit = volatility_data[["volatility_21d", "next_month_volatility"]].quantile(0.995).max()
ax.plot([0, limit], [0, limit], color="#52514e", linestyle="--", linewidth=1)
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel("This month's realised volatility")
ax.set_ylabel("Next month's realised volatility")
ax.set_title("Volatility persists — the same chart for returns is a shapeless cloud")
ax.grid(axis="x")
plt.show()

correlation = volatility_data["volatility_21d"].corr(volatility_data["next_month_volatility"])
print(f"Correlation between this month's and next month's volatility: {correlation:.3f}")

Compare that with the largest feature-to-return correlation from Notebook 03,
which was about 0.08. This is an order of magnitude stronger, and it is
visible to the naked eye.

### The baseline that matters

A regression model here will report a high $R^2$ and look impressive. Do not
be impressed yet. The honest benchmark is not "predict the average" — it is
**persistence**: predict that next month's volatility equals this month's.

That naive rule is free, requires no model, and is hard to beat. If the
machine learning model cannot beat it, the model has added nothing.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score

regressors = {
    "Persistence (no model)": "persistence",
    "Train-set mean (no model)": "constant",
    "Linear regression": LinearRegression(),
    "Ridge regression": Ridge(alpha=10.0),
    "Random forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

rows = []
for name, estimator in regressors.items():
    for fold, (train_index, test_index) in enumerate(splitter.split(months), start=1):
        train = volatility_data[volatility_data["month"].isin(months[train_index])]
        test = volatility_data[volatility_data["month"].isin(months[test_index])]
        if len(test) == 0:
            continue

        y_test = test["next_month_volatility"].to_numpy()

        if estimator == "persistence":
            y_hat = test["volatility_21d"].to_numpy()
        elif estimator == "constant":
            y_hat = np.full(len(test), train["next_month_volatility"].mean())
        else:
            pipeline = Pipeline(
                [("scale", StandardScaler()), ("model", estimator)]
            ).fit(train[FEATURES], train["next_month_volatility"])
            y_hat = pipeline.predict(test[FEATURES])

        rows.append(
            {
                "model": name,
                "fold": fold,
                "r2": r2_score(y_test, y_hat),
                "mae": mean_absolute_error(y_test, y_hat),
            }
        )

volatility_results = pd.DataFrame(rows)
volatility_summary = volatility_results.groupby("model").agg(
    mean_mae=("mae", "mean"),
    worst_fold_mae=("mae", "max"),
    mean_r2=("r2", "mean"),
).sort_values("mean_mae")
volatility_summary.round(4)

In [ ]:
# Per-fold detail, because the averages hide a lot
per_fold = volatility_results.pivot(index="model", columns="fold", values="mae")
per_fold = per_fold.loc[volatility_summary.index]
print("Mean absolute error by fold (lower is better)\n")
print(per_fold.round(4).to_string())

forest_mae = per_fold.loc["Random forest"]
persistence_mae = per_fold.loc["Persistence (no model)"]
wins = int((forest_mae < persistence_mae).sum())
print(f"\nRandom forest beats persistence in {wins} of {len(forest_mae)} folds.")
print(f"Mean improvement in MAE: {1 - forest_mae.mean() / persistence_mae.mean():.1%}")

In [ ]:
plot_order = volatility_summary.sort_values("mean_mae", ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#eb6834" if "no model" in name else "#2a78d6" for name in plot_order.index]
ax.barh(plot_order.index, plot_order["mean_mae"], color=colors, height=0.6)
ax.set_title("Mean absolute error predicting next month's volatility — lower is better")
ax.set_xlabel("Mean absolute error (annualised volatility points)")
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
for y, value in enumerate(plot_order["mean_mae"]):
    ax.text(value + 0.0009, y, f"{value:.4f}", va="center", fontsize=9, color="#52514e")
ax.set_xlim(0, plot_order["mean_mae"].max() * 1.2)
plt.show()

### What this table does and does not say

The random forest beats both free baselines, in four folds out of five, by
about 9% in mean absolute error. That is a **real but modest** edge, and it is
worth stating in exactly those words rather than dressing it up.

Now look at the `mean_r2` column: it is around zero and often negative. A
negative out-of-sample $R^2$ means the model did worse than predicting the
test block's own average — which nobody could have known in advance, so it is
a brutally hard benchmark. On 48 months with big shifts in the market's
volatility level between folds, $R^2$ is unstable and mostly measures those
shifts. **Mean absolute error against a named baseline is the more honest
summary here**, and it tells a consistent story that $R^2$ does not.

Two further things worth noticing:

- **Plain linear regression is the worst model on the list**, worse than
  doing nothing. With 12 correlated features and a few hundred training rows
  it chases noise. Ridge is the same model with a penalty on large
  coefficients, and that one change moves it from worst to second-best.
- **Fold 2 is bad for everything.** That block covers the second half of 2015,
  with the August 2015 volatility spike. Every model trained on the calm
  period before it was surprised. That is what regime change does to a model,
  and it is why risk models are monitored rather than trusted.

Compare the two tasks honestly:

| | Direction | Volatility |
|---|---|---|
| Correlation of best single feature with target | 0.08 | 0.41 |
| Out-of-sample AUC / baseline improvement | ~0.49 (none) | ~9% better MAE |
| Beats the free alternative? | No | Yes, in 4 of 5 folds |

Not "one fails and one triumphs" — one has no signal at all, and the other
has a modest one you can actually use.

### Where does this actually get used?

Volatility forecasts are not an academic exercise. They feed:

- **Value at Risk and Expected Shortfall** — the regulatory capital numbers
  from Notebook 02
- **Position sizing and volatility targeting** — scaling exposure down when
  risk rises
- **Option pricing and hedging** — the volatility input is the whole game
- **Margin models** — CCPs size initial margin off volatility forecasts

This is the pattern across finance. Machine learning earns its keep on
**risk, operations, fraud, credit and client analytics** — problems with
stable relationships and no adversary competing them away. It struggles on
**price direction**, where the relationships are weak by construction and
every participant is trying to find them first.

### Exercise 2

Which features does the volatility model rely on? Fit the random forest
regressor on the full dataset and use `permutation_importance` (as in
Notebook 04) to rank them. Is the VIX pulling its weight, or is a stock's own
trailing volatility doing all the work?

In [ ]:
# YOUR CODE HERE

### Exercise 3

Build a third target of your own and evaluate it the same way — with a named
naive baseline that the model must beat. Suggestions:

- next month's **trading volume** (also persistent)
- whether next month's return is in the worst decile (a crash-risk classifier;
  note the severe class imbalance and use the metrics from Notebook 04)
- next month's **absolute** return, ignoring sign

State your baseline before you fit anything.

In [ ]:
# YOUR CODE HERE

---

## Recap

1. Always name the naive baseline first: the base rate for classification,
   persistence for a time series. Report the improvement over it, not the raw
   score.
2. A classifier that has learned the base rate looks accurate and knows
   nothing. Check what share of predictions are in one class.
3. AUC near 0.5 means no signal, whatever the accuracy says.
4. Put a standard error on any claimed improvement before believing it.
5. Direction is hard because it is competed away. Risk is easier because it
   is not — but "easier" here means a 9% error reduction, not a crystal ball.
6. Out-of-sample $R^2$ can be negative and is unstable under regime shifts.
   Pick the metric that answers the question you actually asked.
7. Regularisation matters: ridge beat plain linear regression decisively.
8. Reporting a clean null result is a professional skill.

---

## Where to take this next

Ideas for a term project, roughly in order of difficulty:

- Add fundamentals (P/E, leverage, profitability) and retest direction.
- Replace the point forecast of volatility with a GARCH(1,1) and compare.
- Build the crash-risk classifier from Exercise 3 and set its threshold from
  an explicit cost of being wrong, as in Notebook 04.
- Take the credit model from Notebook 04 and write the model card: intended
  use, training data, fair-lending testing, monitoring plan.
- Estimate what trading costs would do to the Task A backtest, and find the
  cost level at which the strategy breaks even.